<div dir="rtl">
<h1>ادامهٔ آموزش، ادامهٔ همان ساعت است</h1>
<p>درس 56 از 76 · آیا اندازهٔ گام باید تا آخر ثابت بماند؟ · <code dir="ltr">49b-schedule</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49b-schedule.html">📖 بازگشت به همین درس</a></p>
<p>یک نرخ کسینوسی با افق ثابت بنویسید و خطای شروع دوبارهٔ شمارنده را پیدا کنید.</p><p>پیش‌نیاز: نرخ پایه، شمارهٔ گام از ۱، Warmup و کسینوس درس.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>با پایان کاهش در گام ۶، آیا توقف در گام ۳ باید نرخ گام ۴ را عوض کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
from mini_gpt.schedule import ScheduleConfig
schedule = ScheduleConfig('cosine',2,6,0.1)
print('reference step 1:',schedule.learning_rate(0.001,1))
print('reference step 6:',schedule.learning_rate(0.001,6))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>cosine_rate(Step, peak, Warmup, end, ratio) را برای Step>=1 و end>Warmup>=0 بنویسید. Warmup تا خود گام Warmup است؛ پس از end نرخ روی peak*ratio می‌ماند. حالت Warmup=0 نیز معتبر است.</p>
</div>

In [ ]:
def cosine_rate(step, peak, warmup, end, ratio):
    # TODO: نرخ همان شمارهٔ گام، بدون بازتنظیم افق
    return None

In [ ]:
def test_exercise():
    result = cosine_rate(1,0.001,2,6,0.1)
    if result is None:
        return False
    for warmup,end in ((2,6),(0,6),(1,4)):
        oracle = ScheduleConfig('cosine',warmup,end,0.1)
        for step in (1,2,4,6,9):
            assert math.isclose(cosine_rate(step,0.001,warmup,end,0.1),oracle.learning_rate(0.001,step),abs_tol=1e-12)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: cosine_rate')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط طول Warmup را از ۲ به ۴ تغییر دهید؛ اوج و پایان کاهش ثابت‌اند.</p>
</div>

In [ ]:
for warmup in (2,4):
    trial = ScheduleConfig('cosine',warmup,8,0.1)
    print(warmup,[round(trial.learning_rate(0.001,s),6) for s in range(1,10)])

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>پس از سه گام، نرخ‌های ادامه را اشتباهاً از شمارهٔ ۱ گرفته‌ایم. resumed_rates(schedule, peak, completed, count) باید نرخ count گام بعد از completed را برگرداند.</p>
</div>

In [ ]:
wrong = [schedule.learning_rate(0.001,s) for s in range(1,4)]
expected_steps = [4,5,6]
print('restarted rates:',wrong,'but next step numbers are:',expected_steps)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def resumed_rates(schedule, peak, completed, count):
    # TODO: completed گام انجام شده است
    return None

In [ ]:
def test_repair():
    result = resumed_rates(schedule,0.001,3,3)
    if result is None:
        return False
    assert result == [schedule.learning_rate(0.001,s) for s in (4,5,6)]
    assert resumed_rates(schedule,0.001,0,1)==[0.0005]
    assert resumed_rates(schedule,0.001,6,0)==[]
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: resumed_rates')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>ScheduleConfig همان کلاس مصرف‌شده در train.py است. پایان کاهش از Checkpoint بازیابی می‌شود و به --steps تازه وابسته نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>برابری فرمول نرخ چرا به‌تنهایی برتری کیفیت آموزش را ثابت نمی‌کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49b-schedule.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/49b-schedule.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>